# Medical Continued Pretraining (CPT) Pipeline
## Qwen2.5-7B on Augmented Medical Data

This notebook trains Qwen2.5-7B on medical CPT data for domain adaptation.

**Hardware Requirements:**
- GPU with ≥24GB VRAM (A100, RTX A6000, RTX 4090, or similar)
- Ubuntu 20.04+ with CUDA 12.1+
- 100GB+ free disk space for checkpoints

## 1. Setup and Configuration

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check required packages
try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        DataCollatorForLanguageModeling
    )
    from datasets import Dataset
    import evaluate
    print("\n✓ All required packages available")
except ImportError as e:
    print(f"\n✗ Missing package: {e}")
    print("\nInstall with:")
    print("pip install transformers datasets evaluate")
    sys.exit(1)

### Configuration

In [ ]:
# ============ TRAINING CONFIG ============
config = {
    # Model & Data
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",
    
    # Training Parameters
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,  # Adjust based on GPU memory
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 4,  # Effective batch size = 4 * 4 = 16
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    
    # Sequence & Optimization
    "max_seq_length": 2048,  # Context window
    "bf16": True,  # Mixed precision (bfloat16 for newer GPUs)
    "fp16": False,  # Use bf16 instead on modern GPUs
    
    # Checkpointing & Saving
    "output_dir": "medical_qwen_cpt",
    "save_strategy": "steps",
    "save_steps": 200,
    "save_total_limit": 5,  # Keep only 5 most recent checkpoints
    
    # Evaluation
    "eval_strategy": "steps",
    "eval_steps": 100,
    "metric_for_best_model": "eval_loss",
    
    # Logging
    "logging_dir": "logs",
    "logging_steps": 10,
    "log_level": "info",
    
    # Other
    "seed": 42,
    "dataloader_num_workers": 4,
    "dataloader_pin_memory": True,
}

print("📋 TRAINING CONFIGURATION")
print("=" * 50)
for key, value in config.items():
    print(f"{key:.<40} {value}")
print("=" * 50)

## 2. Load and Prepare Data

In [ ]:
def load_jsonl(file_path: str) -> List[Dict]:
    """Load JSONL file."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Line {i} invalid JSON: {e}")
    return data

# Load data
print("📂 Loading data...")
train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f"✓ Train: {len(train_data):,} chunks")
print(f"✓ Eval:  {len(eval_data):,} chunks")

# Calculate total tokens
train_tokens = sum(c.get('token_count', 0) for c in train_data)
eval_tokens = sum(c.get('token_count', 0) for c in eval_data)
print(f"\n📊 Tokens:")
print(f"  Train: {train_tokens:,}")
print(f"  Eval:  {eval_tokens:,}")

In [ ]:
# Show sample
print("\n📝 Sample training chunk:")
sample = train_data[0]
print(f"\nSource: {sample.get('metadata', {}).get('source_book', 'N/A')}")
print(f"Augmentation: {sample.get('metadata', {}).get('augmentation', 'unknown')}")
print(f"Tokens: {sample.get('token_count')}")
print(f"\nText (first 500 chars):")
text = sample.get('text', '')
print(text[:500] + "..." if len(text) > 500 else text)

## 3. Load Tokenizer and Model

In [ ]:
print(f"🔄 Loading tokenizer: {config['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(
    config["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer):,}")
print(f"  Pad token: {tokenizer.pad_token_id}")
print(f"  EOS token: {tokenizer.eos_token_id}")

In [ ]:
print(f"\n🔄 Loading model: {config['model_name']}...")
model = AutoModelForCausalLM.from_pretrained(
    config["model_name"],
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✓ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {num_params/1e9:.2f}B")
print(f"  Device: {model.device}")
print(f"  Dtype: {next(model.parameters()).dtype}")

## 4. Tokenize Data

In [ ]:
def tokenize_function(examples):
    """Tokenize text with truncation and padding."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

print("🔄 Tokenizing data...")

# Convert to Dataset
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})

# Tokenize
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
    desc="Tokenizing train",
)

eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
    desc="Tokenizing eval",
)

print(f"✓ Train: {len(train_dataset):,} samples")
print(f"✓ Eval:  {len(eval_dataset):,} samples")

In [ ]:
# Show tokenized sample
sample = train_dataset[0]
print("\n📊 Tokenized sample:")
print(f"  Input IDs length: {len(sample['input_ids'])}")
print(f"  Attention mask length: {len(sample['attention_mask'])}")
print(f"  First 20 tokens: {sample['input_ids'][:20]}")

## 5. Setup Training

In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM
)

print("✓ Data collator configured")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=config["output_dir"],
    num_train_epochs=config["num_train_epochs"],
    per_device_train_batch_size=config["per_device_train_batch_size"],
    per_device_eval_batch_size=config["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["learning_rate"],
    warmup_steps=config["warmup_steps"],
    weight_decay=config["weight_decay"],
    max_grad_norm=config["max_grad_norm"],
    bf16=config["bf16"],
    fp16=config["fp16"],
    save_strategy=config["save_strategy"],
    save_steps=config["save_steps"],
    save_total_limit=config["save_total_limit"],
    eval_strategy=config["eval_strategy"],
    eval_steps=config["eval_steps"],
    metric_for_best_model=config["metric_for_best_model"],
    logging_dir=config["logging_dir"],
    logging_steps=config["logging_steps"],
    log_level=config["log_level"],
    seed=config["seed"],
    dataloader_num_workers=config["dataloader_num_workers"],
    dataloader_pin_memory=config["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,  # For eval_loss
    push_to_hub=False,
    report_to=["tensorboard"],
)

print("✓ Training arguments configured")
print(f"\nEffective batch size: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print(f"Total training steps: ~{(len(train_dataset) // (config['per_device_train_batch_size'] * config['gradient_accumulation_steps']) + 1) * config['num_train_epochs']}")

## 6. Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer created and ready")

## 7. Train Model

⚠️ **Warning**: This will take several hours on a single GPU. Grab a coffee!

In [ ]:
print("\n" + "="*80)
print("🚀 STARTING TRAINING")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset: {len(train_dataset):,} training, {len(eval_dataset):,} eval")
print(f"Model: {config['model_name']}")
print(f"Epochs: {config['num_train_epochs']}")
print("="*80 + "\n")

# Train
train_result = trainer.train()

print("\n" + "="*80)
print("✓ TRAINING COMPLETE")
print("="*80)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Checkpoint directory: {config['output_dir']}")
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

## 8. Evaluate Final Model

In [ ]:
print("\n🔍 Evaluating best model...")
eval_results = trainer.evaluate()

print("\n📊 Evaluation Results:")
print("=" * 50)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key:.<40} {value:.4f}")
    else:
        print(f"{key:.<40} {value}")
print("=" * 50)

## 9. Save Final Model

In [ ]:
# Save best model
best_model_path = Path(config["output_dir"]) / "best_model"
print(f"💾 Saving best model to {best_model_path}...")

model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"✓ Model saved: {best_model_path}")
print(f"  Size: {sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9:.2f} GB")

## 10. Test Inference

In [ ]:
from transformers import pipeline

print("🧪 Testing inference on trained model...")

# Load trained model for inference
trained_model = AutoModelForCausalLM.from_pretrained(
    str(best_model_path),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

text_gen = pipeline(
    "text-generation",
    model=trained_model,
    tokenizer=tokenizer,
    device=0,
)

# Test prompts
prompts = [
    "The diagnosis of venous insufficiency requires",
    "Duplex ultrasound imaging is essential for",
    "Treatment of varicose veins includes",
]

print("\n📝 Generated Text Samples:")
print("=" * 80)
for i, prompt in enumerate(prompts):
    print(f"\nPrompt {i+1}: {prompt}")
    output = text_gen(
        prompt,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    print(f"Generated: {output[0]['generated_text']}")
    print("-" * 80)

print("\n✓ Inference test complete")

## 11. Training Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("📋 TRAINING SUMMARY")
print("="*80)

import glob
checkpoints = sorted(glob.glob(f"{config['output_dir']}/checkpoint-*"))

print(f"\n✓ Training completed successfully!")
print(f"\n📂 Output Structure:")
print(f"  Best model:        {best_model_path}")
print(f"  Checkpoints:       {len(checkpoints)} checkpoints saved")
print(f"  Latest checkpoint: {checkpoints[-1] if checkpoints else 'None'}")
print(f"  Training logs:     {config['logging_dir']}/")

print(f"\n📊 Final Metrics:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print(f"\n🚀 Next Steps:")
print(f"  1. Review training curves: tensorboard --logdir {config['logging_dir']}")
print(f"  2. Load best model:  model = AutoModelForCausalLM.from_pretrained('{best_model_path}')")
print(f"  3. Deploy to production or push to Hub")
print(f"  4. Run evaluation: python diagnose_training.py")

print("\n" + "="*80)

## Appendix: Advanced Configuration Guide

### GPU Memory Optimization

If you run out of memory, try these adjustments:

```python
# Reduce batch size
config["per_device_train_batch_size"] = 2  # was 4
config["gradient_accumulation_steps"] = 8  # was 4

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Reduce sequence length
config["max_seq_length"] = 1024  # was 2048
```

### Multi-GPU Training

To use multiple GPUs, add to TrainingArguments:

```python
training_args = TrainingArguments(
    ...
    distributed_type="torch",
    ddp_find_unused_parameters=False,
)
```

Then run with:
```bash
accelerate launch medical_cpt_training.py
```

### Monitoring Training

View training curves in real-time:
```bash
tensorboard --logdir logs
# Open http://localhost:6006 in browser
```

### Common Issues

**Out of Memory Error:**
- Reduce `per_device_train_batch_size`
- Enable gradient checkpointing
- Reduce `max_seq_length`

**Loss not decreasing:**
- Check learning rate (try 1e-5 to 5e-5)
- Increase warmup steps
- Check data quality and format

**Training too slow:**
- Increase `per_device_train_batch_size`
- Reduce `eval_steps` and `save_steps`
- Use more GPUs with distributed training

### Loading Trained Model Later

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("medical_qwen_cpt/best_model")
tokenizer = AutoTokenizer.from_pretrained("medical_qwen_cpt/best_model")

# Use for inference
inputs = tokenizer("The diagnosis of venous insufficiency", return_tensors="pt")
outputs = model.generate(**inputs, max_length=100)
print(tokenizer.decode(outputs[0]))
```